# TactiVision — YOLO11n Fine-tuning
Notebook นี้ใช้เทรนตัวตรวจจับ `player`, `goalkeeper`, `referee`, `ball` บน Google Colab

In [ ]:
!pip install -q "ultralytics>=8.3,<9" pyyaml

## ตั้งตำแหน่ง Dataset
Upload Dataset ที่ Export จาก Roboflow แล้วแก้ `DATA_YAML` ให้ชี้ไปยังไฟล์ `data.yaml`

In [ ]:
from pathlib import Path
import yaml
from ultralytics import YOLO

DATA_YAML = Path('/content/dataset/data.yaml')
MODEL = 'yolo11n.pt'
EXPECTED = {'player', 'goalkeeper', 'referee', 'ball'}

assert DATA_YAML.is_file(), f'ไม่พบ {DATA_YAML}'
config = yaml.safe_load(DATA_YAML.read_text(encoding='utf-8'))
names = config['names']
class_names = list(names.values()) if isinstance(names, dict) else list(names)
assert {str(name).lower() for name in class_names} == EXPECTED, class_names
print('Dataset พร้อม:', class_names)

## Smoke Test 3 Epoch
รันส่วนนี้ก่อนเพื่อตรวจว่า Dataset และ GPU ใช้งานได้

In [ ]:
smoke_model = YOLO(MODEL)
smoke = smoke_model.train(
    data=str(DATA_YAML), epochs=3, imgsz=640, batch=8,
    project='/content/tactivision_runs', name='smoke',
    seed=42, deterministic=True, exist_ok=True
)

## เทรนจริง 50 Epoch
รันเมื่อ Smoke Test ผ่านแล้ว

In [ ]:
model = YOLO(MODEL)
trained = model.train(
    data=str(DATA_YAML), epochs=50, imgsz=640, batch=8,
    project='/content/tactivision_runs', name='tactivision-yolo11n',
    seed=42, deterministic=True, patience=15, exist_ok=True
)
best_path = Path(trained.save_dir) / 'weights' / 'best.pt'
print('โมเดลที่ต้องส่งให้คนที่ 5:', best_path)

## ทดสอบบน Test Set

In [ ]:
best_model = YOLO(str(best_path))
metrics = best_model.val(data=str(DATA_YAML), split='test')
print({
    'precision': metrics.box.mp,
    'recall': metrics.box.mr,
    'mAP50': metrics.box.map50,
    'mAP50-95': metrics.box.map,
})

In [ ]:
from google.colab import files
files.download(str(best_path))